## Ingestion

In [4]:
import requests

def fetch_url(url):
    try:
        response = requests.get(url, timeout=10, verify=False)  # Set a timeout for the request
        response.raise_for_status()  # Raise an error for bad responses
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return None

In [5]:
data = fetch_url("https://medium.com/@manavg/the-definitive-guide-to-designing-effective-agentic-ai-systems-4c7c559c3ab3")

/Users/sai.rachakonda/my_files/notebooks/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'medium.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [6]:
from bs4 import BeautifulSoup

def preprocess_text(text):
    soup = BeautifulSoup(text, "lxml")

    # Remove content that isn't useful for the LLM
    for tag in soup([
        "script",
        "style",
        "noscript",
        "nav",
        "footer",
        "header",
        "aside"
    ]):
        tag.decompose()

    # Extract visible text
    text = soup.get_text(separator=" ", strip=True)

    # Normalize whitespace
    text = " ".join(text.split())

    return text


In [7]:
text = preprocess_text(data)
text[:1000] 

'Medium The Definitive Guide to Designing Effective Agentic AI Systems | by Manav Gupta | Medium Sitemap Open in app Sign up Sign in Medium Logo Get app Write Search Sign up Sign in Manav Gupta Manav Gupta Agentic Ai LLM AI Agent AI The Definitive Guide to Designing Effective Agentic AI Systems Manav Gupta 10 min read · May 2, 2025 -- 3 Listen Share Introduction Agentic AI systems represent a significant evolution — moving beyond the capabilities of standalone large language models (LLMs) to create intelligent systems that can reason, plan, and interact with the world through tools. These systems combine the reasoning power of LLMs with specialized tools that enable them to perform concrete actions. The emergence of agentic AI systems addresses fundamental limitations of traditional AI approaches: 1. Bridging Reasoning and Action : While LLMs excel at reasoning, language understanding, and generation, they cannot directly interact with external systems. Agentic AI bridges this gap by p

## Vector DB initialization

In [ ]:
import chromadb
from chromadb.config import Settings

chroma_client = chromadb.PersistentClient(
    path="./chroma"
)

In [14]:
collection = chroma_client.get_or_create_collection(name="basic_rag", metadata={"hnsw:space": "cosine"})

0

# chunking

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document



text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
all_splits = text_splitter.split_text(text)

In [25]:
len(all_splits)

224

In [26]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
embeddings = OllamaEmbeddings(model="nomic-embed-text")


vector_store = Chroma(
    collection_name="basic_rag",
    embedding_function=embeddings,
    persist_directory="./chroma",  
)

In [27]:
vector_store.add_documents(documents=[Document(page_content=split) for split in all_splits])

['216d7ee0-d2fd-4148-ae1f-a8f1119e4a06',
 'b12912a3-eebc-42e5-9c05-68cc59712c57',
 'b815231d-20e1-46e0-bcfa-c3a3963470df',
 '7af796ae-b584-4189-a85e-7854dba16b27',
 '1783adb5-8c52-45a3-859a-e0a40236fe8e',
 'cab189d4-5b62-4706-9e8a-506c85ac2cb0',
 'eca4ccc1-f828-420b-9fe7-bb4bfd7455d3',
 'adf905a0-b43e-4562-8898-dc87298e7b69',
 '2b679577-7610-422c-bf5d-042617a229b5',
 'ed304efc-5090-4fbd-bc13-2c1b5a5eb688',
 'b6112726-a10c-4653-8db4-845a350b5e70',
 '7f43d5b6-5d5b-43a6-ba6c-47c0d02d8c15',
 'e4b8febf-ebff-4722-ac4d-f8568423cf32',
 '85169cfd-036b-4d46-bf26-326173340397',
 'f530a6a5-4b67-410e-9e26-30c36602e764',
 '283bea32-f4ed-41d2-907b-3539d6516988',
 '756eb514-555f-4b9c-b116-62750b5a5d4d',
 '2413940c-57fb-4b4d-a744-d2fb49554974',
 '07e7d098-0d74-4674-8761-fd411fe64bba',
 'b6c2eb40-e0b9-43a4-a2ce-4e8d4702db87',
 '7f3473b7-6014-42fd-9b3d-f9865993bae9',
 'e4013caa-ba12-4940-9ddb-c4d5bd20e0c2',
 'c183c938-f9e1-423a-a571-767bb078b9cc',
 'c78718b6-32c1-4214-b64e-11c883126c5c',
 '26ce31ce-4813-

In [28]:
similar_docs = vector_store.similarity_search("What is the main idea of the article?", k=3)

# Retrieval

In [40]:
retriever = vector_store.as_retriever()

In [46]:
from langchain_ollama import ChatOllama

from langchain_core.prompts import PromptTemplate 

docs = retriever.invoke("what is the blog about")
docs = '\n'.join([doc.page_content for doc in docs])
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="You are a helpful assistant. Use the following context to answer the question.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"
)



llm = ChatOllama(model="qwen3:latest", model_kwargs={"n": 1, "temperature": 0.7})




[Document(id='c10b2cb6-584b-4c2e-a4a3-f814e5d40d1c', metadata={}, page_content='· 3 following Help Status About Careers Press Blog Store Privacy Rules Terms Text to speech'),
 Document(id='be3d1add-2c7b-4d22-badc-f0082feb828a', metadata={}, page_content='information that requires judgment. 5. Interact with Users : Managing the dialogue and explaining'),
 Document(id='b147e417-d235-47e3-9315-44cdbf1ffad2', metadata={}, page_content=': Defined with clear descriptions (for example, YAML configuration in crewAI implementation)'),
 Document(id='2227c818-a9bb-4b58-a086-11b7ab288899', metadata={}, page_content='the rationale for its decisions, often referencing the types of patterns (pairs, directory groups)')]

[Document(id='3234bf84-b02e-40a1-a37c-3d7bceb6c8b3', metadata={}, page_content='Design Principles Purpose : Tools perform specific, well-defined technical operations. They are the'),
 Document(id='b3eed5ee-ca1c-485a-a2c1-33a0ca14b41a', metadata={}, page_content='They are the system’s means of interacting with data and the external world. Characteristics: Focus'),
 Document(id='20006686-607b-4eaa-b452-4f1f6f5013de', metadata={}, page_content='the coherence of PR groups, generating titles and descriptions, or providing a rationale for')]